## Cross-Protocol Comparison — Protocol A vs Protocol B

Runs `experiment.md`'s paired sweep on the `costream` pipeline: Protocol A (presegmented) vs Protocol B (streaming) evaluation, across the fixed grid — 5 models × 5 folds × 7 window sizes × 3 seeds — over the 32 training-cohort participants only (`results/subjects.json`). The 9-participant hold-out cohort is never loaded in this notebook.

**Read `CLAUDE.md` and `experiment.md` before changing anything below** — the segmentation, the gate, the scoring formula, and the split structure are specified there and must not be re-derived here.

**Known issue — `ResNet` is excluded from `MODEL_NAMES` below.** It hangs mid-fit when run in a fresh Python process on this machine, even with `tensorflow-metal` GPU acceleration enabled and confirmed active (TF picks up the M1 GPU in the logs, then the fit call itself stalls indefinitely — verified on trivially small data, CPU time frozen for 10+ minutes). It has fit successfully before in this exact repo (`results/cv_all_windows.csv` has 35 completed ResNet fits, produced from a long-lived Jupyter kernel), so this looks like an invocation-context issue rather than a fundamental incompatibility. Needs separate investigation before re-enabling — do not silently re-add it without resolving that first.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import sys
import time
from pathlib import Path

import pandas as pd

# Jupyter sets cwd to this notebook's own directory (experiments/), not the repo
# root — anchor on CLAUDE.md so paths below are correct regardless of how/where
# this notebook is launched from.
def _find_repo_root(marker="CLAUDE.md"):
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root (no {marker} found above {p})")

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from costream.data import load_farseeing_subject_map
from costream.evaluation import ModelSpec, run_subject_cv, assert_adl_exclusion_zone
from scripts.models import get_model_specs

### Experiment Grid

In [ ]:
WINDOW_SIZES = [3, 5, 7, 10, 15, 30, 60]  # seconds
SEEDS = [0, 1, 2]
CV_FOLDS = 5
MODEL_NAMES = ["Catch22", "ExtraTrees", "MiniRocket", "QUANT"]  # ResNet excluded, see note above

# Fixed for this experiment (experiment.md: "a tuned threshold would confound
# protocol with operating point"). None of these models set a `threshold_`
# attribute, so evaluate_models/evaluate_presegmented's
# getattr(model, "threshold_", 0.5) already falls back to exactly this —
# asserted below rather than assumed.
FIXED_TAU = 0.5

DATA_PATH = ROOT / "data/farseeing.feather"
SUBJECTS_PATH = ROOT / "results/subjects.json"
RAW_CSV = ROOT / "results/cross_protocol_raw.csv"
FP_CSV = ROOT / "results/protocol_b_false_positives.csv"
CACHE_DIR = ROOT / "results/prob_cache"

print(f"Window sizes: {WINDOW_SIZES}")
print(f"Seeds: {SEEDS}")
print(f"Models: {MODEL_NAMES}")

### Load Data & Training Cohort

The hold-out cohort (9 participants) is never loaded here — only `results/subjects.json`'s `train` list.

In [ ]:
train_subjects = json.load(open(SUBJECTS_PATH))["train"]
print(f"Training cohort: {len(train_subjects)} participants (hold-out cohort not loaded)")

full_subject_map = load_farseeing_subject_map(DATA_PATH)
subject_map = {s: full_subject_map[s] for s in train_subjects if s in full_subject_map}
assert len(subject_map) == len(train_subjects), "some training subjects missing from data"
n_signals = sum(len(v) for v in subject_map.values())
print(f"{len(subject_map)} subjects, {n_signals} signals")

### Model Specs

Reuses the already-validated `scripts.models` spec factory (same hyperparameters as every prior published result in this repo) rather than re-deriving classifier construction here.

In [ ]:
def build_model_specs(seed: int):
    legacy_specs = get_model_specs(subset=MODEL_NAMES, random_state=seed)
    return [ModelSpec(name=s.name, estimator=s.estimator) for s in legacy_specs]

### Diagnostic (optional) — ADL exclusion-zone check

**Currently known to fail** — `create_training_data`'s actual exclusion zone is `[f - max(pre_offsets), f + w)`, narrower than the `[f-(w+20), f+20)` `CLAUDE.md` declares. This cell reports the gap without halting the notebook; it is not wired into the sweep below. See the change-request discussion for the full explanation before deciding whether to change `training_segmenter.py`.

In [ ]:
sample_dfs = [df for recs in list(subject_map.values())[:10] for df in recs]
for w in [3, 60]:
    try:
        assert_adl_exclusion_zone(sample_dfs, "mag", "label", window_size=float(w), tolerance=20.0)
        print(f"w={w}: PASSED (no violations)")
    except AssertionError as e:
        print(f"w={w}: FAILED -> {e}")

### Sweep Helpers (incremental, resumable)

Writes `results/cross_protocol_raw.csv` and `results/protocol_b_false_positives.csv` incrementally, one `(window_size, seed)` combination at a time, so a crash mid-run does not lose completed cells. Re-running this notebook skips combinations already present in the raw CSV, and the probability cache under `results/prob_cache` means even a from-scratch rerun of a *new* combination reuses any already-cached per-window scores.

In [ ]:
def already_done(window_size: float, seed: int) -> bool:
    if not RAW_CSV.exists():
        return False
    existing = pd.read_csv(RAW_CSV, usecols=["window_size", "seed"])
    return ((existing["window_size"] == window_size) & (existing["seed"] == seed)).any()


def append_rows(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

### Run the Sweep

In [ ]:
total_combos = len(WINDOW_SIZES) * len(SEEDS)
done = 0
t_start = time.time()

for window_size in WINDOW_SIZES:
    for seed in SEEDS:
        done += 1
        if already_done(window_size, seed):
            print(f"[{done}/{total_combos}] window_size={window_size} seed={seed}: already in {RAW_CSV.name}, skipping")
            continue

        print(f"[{done}/{total_combos}] window_size={window_size} seed={seed}: running...")
        t0 = time.time()

        model_specs = build_model_specs(seed)
        results = run_subject_cv(
            subject_map,
            model_specs,
            feature_cols=["mag"],
            label_col="label",
            cv=CV_FOLDS,
            random_state=seed,
            window_size=float(window_size),
            step=1.0,
            freq=100,
            activity_threshold=1.4,
            spacing="multiphase",
            tolerance=20.0,  # t in R = [f-1, f+t); see CLAUDE.md Protocol B "Scoring"
            debounce_secs=60.0,
            verbose=False,
            cache_dir=CACHE_DIR,
        )

        bad_tau = results.loc[results["thresh"] != FIXED_TAU, ["model", "protocol", "thresh"]]
        if len(bad_tau):
            raise RuntimeError(
                f"Expected fixed tau={FIXED_TAU} for every row (no tuning in this "
                f"experiment) but found: {bad_tau.to_dict('records')}"
            )

        false_positives = results.attrs.get("false_positives")

        results["window_size"] = window_size
        results["seed"] = seed
        append_rows(RAW_CSV, results)

        if false_positives is not None and len(false_positives):
            false_positives = false_positives.copy()
            false_positives["window_size"] = window_size
            false_positives["seed"] = seed
            append_rows(FP_CSV, false_positives)

        dt = time.time() - t0
        n_fp = 0 if false_positives is None else len(false_positives)
        print(f"  done in {dt:.1f}s ({len(results)} rows appended, {n_fp} false positives logged)")

elapsed = time.time() - t_start
print(f"\nSweep complete in {elapsed/60:.1f} min. Raw results: {RAW_CSV}")

### Summary

In [ ]:
# experiment.md's "35 paired cells per model" is the (fold, window_size) grain used
# by the downstream ranking analysis (step 5) — 3 seed-rows collapse into each cell
# there. This raw file keeps all of them (Deliverables: "one row per model x fold x
# window_size x protocol x seed"), so check both counts.
expected_cells = CV_FOLDS * len(WINDOW_SIZES)
expected_rows = expected_cells * len(SEEDS)
final = pd.read_csv(RAW_CSV)
for protocol in ["A", "B"]:
    sub = final[final["protocol"] == protocol]
    rows = sub.groupby("model").size()
    cells_ = sub.groupby("model").apply(lambda g: g[["fold", "window_size"]].drop_duplicates().shape[0])
    print(f"\nProtocol {protocol} (expect {expected_rows} raw rows / {expected_cells} unique cells per model):")
    print(pd.DataFrame({"rows": rows, "unique_cells": cells_}))